# City tile generator

Turns an area of a city into 3D-printable tiles for the physical table model.

**How to use:** run the sections in order, top to bottom. Each cell has a ▶ button on its left.
Only the options in section 1 are meant to be edited. The other cells show just their title; double-click one to see its code.

| Section | What it does | Time |
|---|---|---|
| 1. Options | Model scale, tile size, magnets and area settings | – |
| 2. Setup | Installs the tools (first run of a session only) | 2–4 min |
| 3. Area selection | Draw the model area on a map | – |
| 4. Layout preview | Downloads buildings and shows the tile grid | ~1 min |
| 5. Generate tiles | Builds the 3D city and cuts it into tiles | several min |
| 6. Magnet holes | Adds magnet pockets to the tiles (optional) | ~1 min per tile |
| 7. Download | Downloads all results as one zip | – |

Changed an option? Re-run from section 3.

## 1. Options
The defaults match the printed base tiles, so usually nothing here needs changing. Magnet or tile changes must be matched in the base tiles (`base_tiles/base_geometry.gh`). With ALIGN, upload the old `raw_city_mesh.stl` files to `old_meshes/` after section 2.

In [ ]:
# Model
SCALE = 1250                  # Model scale 1:N
TILE_SIZE = 0.2               # Printed tile size [m]
BUILDING_MARGIN = 50          # Extra area meshed around the model [m] (not used with ALIGN)

# Magnets and base tiles [mm]
MAGNET_RADIUS, MAGNET_HEIGHT = 5, 2   # Magnet hole radius and pocket depth
MAGNET_PERIMETER_OFFSET = 8           # Tile edge to magnet centre
BASE_TILE_THICKNESS = 20              # Tile depths snap to this step
ANCHOR_Z_LEVEL = 0                    # Z level the steps are counted from [m]

# Manual bounds: type the area instead of drawing it (SWEREF99 TM, SW corner, size in m)
MANUAL_BOUNDS = False
ORIGIN_X, ORIGIN_Y = 319_470.0, 6_397_410.0
SIZE_X, SIZE_Y = 750, 1250

# Extend an existing model: snap to its grid (SW node)
ALIGN = False
ALIGNMENT_NODE_X, ALIGNMENT_NODE_Y = 319_470.0, 6_397_410.0

## 2. Setup
Installs the Python packages and Blender, and loads the helper code from the repository. This takes a few minutes the first time and is quick when re-run in the same session.

After this cell has run you can upload old meshes to `old_meshes/` in the file browser (folder icon on the left), if you use ALIGN.

In [ ]:
#@title Install and load tools { display-mode: "form" }
# The code lives next to this notebook in city_tiles/:
#   tile_setup.py           package and Blender installation
#   tile_pipeline.py        area selection, meshing and tiling functions
#   blender_magnet_holes.py Blender script used in section 6
import os, sys, subprocess

# dtcc-core is installed from GitHub and its API changes often, so it is pinned
# to a specific commit. Update this hash deliberately and re-test the notebook.
DTCC_CORE_COMMIT = "f8ab79b7b589759a7f9279eac576a725f722ed6a"  # 2026-11-02
REPO_BRANCH = "slim-notebook"  # Branch to fetch the helper code from when running on Colab

if not os.path.exists("tile_pipeline.py"):  # On Colab: fetch the helper code
    if not os.path.isdir("mrstudio-printing"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH,
                        "https://github.com/SB-Chalmers/mrstudio-printing"], check=True)
    sys.path.insert(0, os.path.abspath("mrstudio-printing/city_tiles"))

import tile_setup
tile_setup.setup(DTCC_CORE_COMMIT)

import tile_pipeline as tp
print("Setup done.")

## 3. Area selection
Run the cell, then use the tools on the left of the map:

1. **Rectangle (blue):** draw the model area first. It is expanded to whole tiles.
2. **Polygon (red):** optionally draw around buildings to leave out, for example ones you will replace with detailed models. A building is left out only if the shape covers all of it.

The size and area of each shape are printed under the map. To start over, run the cell again. With *Manual bounds* ticked, every shape you draw is an exclusion.

In [ ]:
#@title Draw the model area { display-mode: "form" }
opts = tp.Options.from_namespace(globals())  # Reads the options from section 1
BOUNDARY_GEOJSON = tp.select_area(opts)

## 4. Layout preview
Downloads the building footprints and shows the model with its tile grid, with left-out buildings in red. Check that the area and the number of tiles look right before going on. The picture is also saved as `layout_preview.png`.

In [ ]:
#@title Preview the layout { display-mode: "form" }
layout = tp.preview_layout(BOUNDARY_GEOJSON, opts)

## 5. Generate tiles
Downloads the point cloud, builds the terrain and buildings, and cuts the city into tiles. This is the slow step; large areas can take a long time. Tiles are saved to `processed_tiles/` as `tile_{i}_{j}.stl`, with `i` counted eastward and `j` northward from the south-west corner.

In [ ]:
#@title Generate the tiles { display-mode: "form" }
mesh = tp.generate_city_tiles(layout, opts)

## 6. Magnet holes (optional)
Adds four magnet pockets to the underside of every tile in `processed_tiles/`, using Blender. Skip this section if you don't use magnets. Mind the magnet orientation when assembling: all magnets should face the same way in city tiles and base tiles.

In [ ]:
#@title Add magnet holes { display-mode: "form" }
tp.add_magnet_holes(opts)

## 7. Download results
Downloads one zip with the tiles, the layout preview and `raw_city_mesh.stl` (keep this one if you may extend the model later). Colab deletes all files when the session ends, so download before closing.

In [ ]:
#@title Download results as a zip { display-mode: "form" }
tp.download_results()